In [1]:
# Evaluation function
from sklearn.metrics import accuracy_score, matthews_corrcoef, roc_auc_score, average_precision_score
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
def evaluate_model(model, X_val = None, X_val_pt5 = None, X_val_PSSM = None, y_val = None, print_metrics=True):
    y_true = y_val

    # Predict probabilities (or logits if using `from_logits=True`).
    if X_val_pt5 is None:
        y_pred_probs = model.predict(X_val)
    elif X_val is None:
        y_pred_probs = model.predict(X_val_pt5)
    elif X_val_PSSM is None:
        y_pred_probs = model.predict([X_val, X_val_pt5])
    else:
        y_pred_probs = model.predict([X_val, X_val_pt5, X_val_PSSM])

    # Convert probabilities/logits to binary predictions (threshold = 0.5).
    y_pred = (y_pred_probs > 0.5).astype(int)

    # If y_true is one-hot encoded, convert it to binary format
    if len(y_true.shape) > 1 and y_true.shape[1] > 1:  # Check if y_true is one-hot encoded
        y_true = np.argmax(y_true, axis=1)  # Convert one-hot encoded y_true to binary labels

    # Ensure y_pred is also 1D
    if len(y_pred.shape) > 1 and y_pred.shape[1] > 1:
        y_pred = np.argmax(y_pred, axis=1)  # Convert y_pred to binary labels if necessary

    # check if only one label is present
    if len(set(y_true)) == 1:
        accuracy = accuracy_score(y_true, y_pred)
        print(f'Accuracy: {accuracy:.4f}')
        print("Only one label is present in the dataset. Cannot compute metrics.")
        return

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred_probs)
    auprc = average_precision_score(y_true, y_pred_probs)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    # Compute Specificity
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)

    # Print the results
    if print_metrics:
        print(f'Accuracy: {accuracy:.4f}')
        print(f'MCC: {mcc:.4f}')
        print(f'AUC: {auc:.4f}')
        print(f'AUPRC: {auprc:.4f}')
        print(f'Precision: {precision:.4f}')
        print(f'Recall: {recall:.4f}')
        print(f'Specificity: {specificity:.4f}')
        print(f'F1 Score: {f1:.4f}')

    return accuracy, mcc, auc, auprc, precision, recall, specificity, f1

In [2]:
# Load ResLysEmbed model
import tensorflow as tf
from keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Concatenate, Lambda
from keras.models import Model
from keras.optimizers import Adam
from keras.losses import BinaryCrossentropy
from keras.initializers import Constant


def residual_block(input_tensor, filters):
    x = Conv1D(filters, kernel_size=3, activation='relu', padding='same')(input_tensor)
    x = Conv1D(filters, kernel_size=3, activation='relu', padding='same')(x)    
    x = Concatenate()([x, input_tensor])  # Skip connection
    return x

def build_resnet_model(input_shape_conv, input_shape_ann):
    # Conv1D branch for sequence data
    conv_input = Input(shape=input_shape_conv)
    x_conv = Embedding(input_dim=21, output_dim=21, name='embedding', trainable=True)(conv_input)

    # Apply Residual blocks
    x_conv = residual_block(x_conv, 32)
    x_conv = MaxPooling1D(pool_size=2)(x_conv)
    x_conv = residual_block(x_conv, 64)
    x_conv = MaxPooling1D(pool_size=2)(x_conv)
    x_conv = Flatten()(x_conv)

    # Dense layer for sequence features
    x_conv = Dense(32, activation='relu')(x_conv)
    x_conv = Dropout(0.3)(x_conv)

    # ANN branch for prot_t5 embeddings
    ann_input = Input(shape=(input_shape_ann,))
    x_ann = Dense(32, activation='relu')(ann_input)
    x_ann = Dropout(0.3)(x_ann)

    # Concatenate Conv1D (ResNet) and ANN branches
    combined = Concatenate()([x_conv, x_ann])

    # Output layer
    x = Dense(32, activation='relu')(combined)
    x = Dropout(0.3)(x)
    output_layer = Dense(1, activation='sigmoid')(x)

    # Build model
    model = Model(inputs=[conv_input, ann_input], outputs=output_layer)
    model.compile(optimizer=Adam(learning_rate=0.0001),
                  loss=BinaryCrossentropy(),
                  metrics=['accuracy'])

    return model


# Define the model with Conv1D input shape (33,) and ANN input shape 1024
ResLysEmbed = build_resnet_model((33,), 1024)
ResLysEmbed.summary()

ResLysEmbed.load_weights('Model/res_model.weights.h5')

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 33)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 33, 21)    │        441 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 33, 32)    │      2,048 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 33, 32)    │      3,104 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 33, 53)    │          0 │ conv1d_1[0][0],   │
│ (Concatenate)       │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 16, 53)    │          0 │ concatenate[0][0] │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 16, 64)    │     10,240 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 16, 64)    │     12,352 │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 16, 117)   │          0 │ conv1d_3[0][0],   │
│ (Concatenate)       │                   │            │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 8, 117)    │          0 │ concatenate_1[0]… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 936)       │          0 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1024)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │     29,984 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │     32,800 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 64)        │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 93,082 (363.60 KB)

 Trainable params: 93,082 (363.60 KB)

 Non-trainable params: 0 (0.00 B)

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\saving\saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 36 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [3]:
def read_protein_site_pairs(filename):
    pairs = []
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('>'):
                parts = line.strip().split('\t')
                protein_id = parts[0][1:]  # remove '>'
                site = int(parts[1])
                pairs.append((protein_id, site))
    return pairs


In [8]:
# Load Train and Test data
import numpy as np
import pandas as pd


def prepare_test_data(fasta_path, DBPTM = False):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY-'
    aa_to_int = {aa: i for i, aa in enumerate(amino_acids)}
    pairs_set = set(read_protein_site_pairs(fasta_path))
    if DBPTM:
        test = pd.read_csv('PTMGPT2/succinylation_benchmark_filtered.csv')
    else:
        test = pd.read_csv('Original dataset/test_t5.csv')
    test = test[test.apply(lambda row: (row['protein_id'], row['site']) in pairs_set, axis=1)]
    test = test[test['sequence'].apply(lambda x: all(aa in amino_acids for aa in x))]
    if DBPTM:
        X_test_embeddings = test['embedding'].apply(lambda x: np.array([float(i) for i in x.strip('[]').split(',')]))
    else:
        X_test_embeddings = test['embedding'].apply(lambda x: np.array([float(i) for i in x.strip('[]').split()]))
    X_test_embeddings = np.stack(X_test_embeddings.values)
    X_test = test['sequence'].values
    y_test = test['label'].values
    X_test_num = np.array([[aa_to_int[aa] for aa in seq] for seq in X_test])

    # print(test.shape)
    # print("X_test_embeddings shape:", X_test_embeddings.shape)
    # print("X_test_num shape:", X_test_num.shape)
    # print("y_test shape:", y_test.shape)
    print("Positive samples in test:", np.sum(y_test == 1))
    print("Negative samples in test:", np.sum(y_test == 0))

    return X_test_num, X_test_embeddings, y_test


In [9]:
import pandas as pd
from sklearn.metrics import accuracy_score, matthews_corrcoef, roc_auc_score, average_precision_score, precision_score, recall_score, confusion_matrix, f1_score
# Read the CSV file
def evaluate_PTMGPT2_predictions(csv_path, fasta_path):
    df = pd.read_csv(csv_path)
    pairs_set = set(read_protein_site_pairs(fasta_path))
    filtered_df = df[df.apply(lambda row: (row['protein_id'], row['site']) in pairs_set, axis=1)]
    y_true = filtered_df['label'].values
    y_pred = filtered_df['prediction'].values
    # check if only one label is present
    if len(set(y_true)) == 1:
        # just calculate accuracy
        accuracy = accuracy_score(y_true, y_pred)
        print(f'Accuracy: {accuracy:.4f}')
        print("Only one label is present in the dataset. Cannot compute other metrics.")
        return
    accuracy = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    auprc = average_precision_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)

    print(f'Accuracy: {accuracy:.4f}')
    print(f'MCC: {mcc:.4f}')
    print(f'AUC: {auc:.4f}')
    print(f'AUPRC: {auprc:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'Specificity: {specificity:.4f}')
    print(f'F1 Score: {f1:.4f}')

def evaluate_ResLysEmbed_predictions(fasta_file, DBPTM=False):
    X_test_num, X_test_embeddings, y_test = prepare_test_data(fasta_file, DBPTM=DBPTM)
    
    # Load the model
    model = ResLysEmbed

    # Evaluate the model
    print("Evaluating ResLysEmbed predictions:")
    evaluate_model(model, X_val=X_test_num, X_val_pt5=X_test_embeddings, y_val=y_test)



In [10]:
print("Evaluating ResLysEmbed predictions with full test set:")
evaluate_ResLysEmbed_predictions('Fasta files/test_original.fasta')

print("Evaluating PTMGPT2 predictions with full test set:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_final.csv', 'Fasta files/test_original.fasta')

print("Evaluating ResLysEmbed predictions with 0.5 sequence identity filter:")
evaluate_ResLysEmbed_predictions('Fasta files/test_filtered_original_0.5.fasta')

print("Evaluating PTMGPT2 predictions with 0.5 sequence identity filter:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_final.csv', 'Fasta files/test_filtered_original_0.5.fasta')

print("Evaluating ResLysEmbed predictions with 0.6 sequence identity filter:")
evaluate_ResLysEmbed_predictions('Fasta files/test_filtered_original_0.6.fasta')

print("Evaluating PTMGPT2 predictions with 0.6 sequence identity filter:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_final.csv', 'Fasta files/test_filtered_original_0.6.fasta')

print("Evaluating ResLysEmbed predictions with 0.7 sequence identity filter:")
evaluate_ResLysEmbed_predictions('Fasta files/test_filtered_original_0.7.fasta')

print("Evaluating PTMGPT2 predictions with 0.7 sequence identity filter:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_final.csv', 'Fasta files/test_filtered_original_0.7.fasta')

print("Evaluating ResLysEmbed predictions with 0.8 sequence identity filter:")
evaluate_ResLysEmbed_predictions('Fasta files/test_filtered_original_0.8.fasta')

print("Evaluating PTMGPT2 predictions with 0.8 sequence identity filter:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_final.csv', 'Fasta files/test_filtered_original_0.8.fasta')

print("Evaluating ResLysEmbed predictions with 0.9 sequence identity filter:")
evaluate_ResLysEmbed_predictions('Fasta files/test_filtered_original_0.9.fasta')

print("Evaluating PTMGPT2 predictions with 0.9 sequence identity filter:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_final.csv', 'Fasta files/test_filtered_original_0.9.fasta')

Evaluating ResLysEmbed predictions with full test set:
Positive samples in test: 253
Negative samples in test: 2973
Evaluating ResLysEmbed predictions:
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Accuracy: 0.8053
MCC: 0.3893
AUC: 0.8733
AUPRC: 0.3482
Precision: 0.2624
Recall: 0.8182
Specificity: 0.8042
F1 Score: 0.3973
Evaluating PTMGPT2 predictions with full test set:
Accuracy: 0.9209
MCC: 0.5921
AUC: 0.8663
AUPRC: 0.4133
Precision: 0.4963
Recall: 0.8016
Specificity: 0.9310
F1 Score: 0.6131
Evaluating ResLysEmbed predictions with 0.5 sequence identity filter:
Positive samples in test: 4
Negative samples in test: 123
Evaluating ResLysEmbed predictions:
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Accuracy: 0.8583
MCC: 0.3940
AUC: 0.9502
AUPRC: 0.2984
Precision: 0.1818
Recall: 1.0000
Specificity: 0.8537
F1 Score: 0.3077
Evaluating PTMGPT2 predictions with 0.5 sequence identity filter:
Accuracy: 0.9528
MCC: 0.2256
AUC: 0.6128
AUPRC: 0.0861
Precision: 0.2500
Recall: 0.2500
Specificity: 0.9756
F1 Sc

In [29]:
print("Evaluating ResLysEmbed predictions with full DBPTM test set:")
evaluate_ResLysEmbed_predictions('Fasta files/DBPTM_test.fasta', DBPTM=True)
print("Evaluating PTMGPT2 predictions with full DBPTM test set:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_DBPTM.csv', 'Fasta files/DBPTM_test.fasta')
print("Evaluating ResLysEmbed predictions with 0.5 sequence identity filter on DBPTM test set:")
evaluate_ResLysEmbed_predictions('Fasta files/DBPTM_test_filtered_0.5.fasta', DBPTM=True)
print("Evaluating PTMGPT2 predictions with 0.5 sequence identity filter on DBPTM test set:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_DBPTM.csv', 'Fasta files/DBPTM_test_filtered_0.5.fasta')
print("Evaluating ResLysEmbed predictions with 0.6 sequence identity filter on DBPTM test set:")
evaluate_ResLysEmbed_predictions('Fasta files/DBPTM_test_filtered_0.6.fasta', DBPTM=True)
print("Evaluating PTMGPT2 predictions with 0.6 sequence identity filter on DBPTM test set:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_DBPTM.csv', 'Fasta files/DBPTM_test_filtered_0.6.fasta')
print("Evaluating ResLysEmbed predictions with 0.7 sequence identity filter on DBPTM test set:")
evaluate_ResLysEmbed_predictions('Fasta files/DBPTM_test_filtered_0.7.fasta', DBPTM=True)
print("Evaluating PTMGPT2 predictions with 0.7 sequence identity filter on DBPTM test set:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_DBPTM.csv', 'Fasta files/DBPTM_test_filtered_0.7.fasta')
print("Evaluating ResLysEmbed predictions with 0.8 sequence identity filter on DBPTM test set:")
evaluate_ResLysEmbed_predictions('Fasta files/DBPTM_test_filtered_0.8.fasta', DBPTM=True)
print("Evaluating PTMGPT2 predictions with 0.8 sequence identity filter on DBPTM test set:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_DBPTM.csv', 'Fasta files/DBPTM_test_filtered_0.8.fasta')
print("Evaluating ResLysEmbed predictions with 0.9 sequence identity filter on DBPTM test set:")
evaluate_ResLysEmbed_predictions('Fasta files/DBPTM_test_filtered_0.9.fasta', DBPTM=True)
print("Evaluating PTMGPT2 predictions with 0.9 sequence identity filter on DBPTM test set:")
evaluate_PTMGPT2_predictions('PTMGPT2/predictions_ptmgpt2_DBPTM.csv', 'Fasta files/DBPTM_test_filtered_0.9.fasta')
print("All evaluations completed.")

Evaluating ResLysEmbed predictions with full DBPTM test set:
Positive samples in test: 1901
Negative samples in test: 2309
Evaluating ResLysEmbed predictions:
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Accuracy: 0.7264
MCC: 0.4444
AUC: 0.7889
AUPRC: 0.7233
Precision: 0.7313
Recall: 0.6228
Specificity: 0.8116
F1 Score: 0.6727
Evaluating PTMGPT2 predictions with full DBPTM test set:
Accuracy: 0.9335
MCC: 0.8693
AUC: 0.9277
AUPRC: 0.9124
Precision: 0.9822
Recall: 0.8685
Specificity: 0.9870
F1 Score: 0.9218
Evaluating ResLysEmbed predictions with 0.5 sequence identity filter on DBPTM test set:
Positive samples in test: 0
Negative samples in test: 1
Evaluating ResLysEmbed predictions:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Accuracy: 1.0000
Only one label is present in the dataset. Cannot compute metrics.
Evaluating PTMGPT2 predictions with 0.5 sequence identity filter on DBPTM test set:
Accuracy: 1.0000
Only one label is present in the dataset. Cannot compute other metrics.
Evaluating ResLysEm